# MERIT-Net — Reproducibility Notebook

Reproduces the main results of the paper: Table 2, Table 3, and Figures 2-6.
Fixed seed = 42; five-model uniform soft-voting ensemble (see `config.json`).

Run order: this notebook is self-contained. On a CPU it takes a few minutes (50 model fits).


## 1. Setup and data


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, os
import matplotlib; import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
                             f1_score, matthews_corrcoef, brier_score_loss, recall_score, precision_score)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

SEED, NTREES = 42, 500
os.makedirs("figures", exist_ok=True)

# load the cleaned dataset (local file, else from the GitHub repo)
PATH = "mendeley_cleaned.csv"
if not os.path.exists(PATH):
    PATH = "https://raw.githubusercontent.com/hussein-alnaffakh1984/merit-net/main/mendeley_cleaned.csv"
df = pd.read_csv(PATH)

FEATURES = ["Age","SystolicBP","DiastolicBP","BS","BodyTemp","BMI","PrevComplications",
            "PreexistDiabetes","GestDiabetes","MentalHealth","HeartRate"]
X = df[FEATURES].values
y = (df["RiskLevel"].astype(str).str.lower() == "high").astype(int).values
print("rows:", len(df), "| class balance:", df["RiskLevel"].value_counts().to_dict())


## 2. Ensemble and 10-fold cross-validation
Five tree-based models with 5:1 cost weighting, combined by uniform soft voting.
Fresh models are created in each fold (no leakage); per-model probabilities are kept for the Trust Score.


In [ ]:
def make_models(n=NTREES):
    return [
        RandomForestClassifier(n_estimators=n, max_depth=20, random_state=SEED, class_weight={0:1.0,1:5.0}),
        XGBClassifier(n_estimators=n, max_depth=10, learning_rate=0.05, scale_pos_weight=5, random_state=SEED, eval_metric="logloss"),
        CatBoostClassifier(iterations=n, depth=12, learning_rate=0.05, class_weights=[1.0,5.0], random_state=SEED, verbose=0),
        LGBMClassifier(n_estimators=n, max_depth=10, learning_rate=0.05, class_weight={0:1.0,1:5.0}, random_state=SEED, verbose=-1),
        ExtraTreesClassifier(n_estimators=n, max_depth=20, random_state=SEED, class_weight={0:1.0,1:5.0}),
    ]

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
ens_p   = np.zeros(len(y))
model_p = np.zeros((len(y), 5))
fold_id = np.empty(len(y), int)
for k,(tr,te) in enumerate(cv.split(X, y)):
    for j,m in enumerate(make_models()):
        m.fit(X[tr], y[tr]); model_p[te, j] = m.predict_proba(X[te])[:, 1]
    ens_p[te] = model_p[te].mean(1); fold_id[te] = k
pred = (ens_p >= 0.5).astype(int)
print("cross-validation complete")


## 3. Table 2 — MERIT-Net performance (10-fold CV, n = 1,164)


In [ ]:
cm = confusion_matrix(y, pred); tn,fp,fn,tp = cm.ravel()
table2 = {
 "Accuracy %":     round(accuracy_score(y, pred)*100, 2),
 "Weighted F1 %":  round(f1_score(y, pred, average="weighted")*100, 2),
 "AUC (ROC)":      round(roc_auc_score(y, ens_p), 4),
 "High-risk recall %":    round(recall_score(y, pred)*100, 2),
 "High-risk precision %": round(precision_score(y, pred)*100, 2),
 "Specificity %":  round(tn/(tn+fp)*100, 2),
 "MCC":            round(matthews_corrcoef(y, pred), 4),
 "Brier":          round(brier_score_loss(y, ens_p), 4),
}
print("Confusion matrix [TN, FP, FN, TP]:", [tn,fp,fn,tp])
for k,v in table2.items(): print(f"  {k:24s}: {v}")


## 4. Figure 2 — Confusion matrix


In [ ]:
fig,ax = plt.subplots(figsize=(4.2,4))
im = ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha="center", va="center",
                color="white" if cm[i,j] > cm.max()/2 else "black", fontsize=14)
ax.set_xticks([0,1]); ax.set_xticklabels(["Low","High"]); ax.set_yticks([0,1]); ax.set_yticklabels(["Low","High"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion matrix (10-fold CV)")
plt.tight_layout(); plt.savefig("figures/fig2_confusion_matrix.png", dpi=300); plt.show()


## 5. Figure 6 — ROC curve


In [ ]:
fpr,tpr,_ = roc_curve(y, ens_p); auc = roc_auc_score(y, ens_p)
plt.figure(figsize=(4.5,4))
plt.plot(fpr, tpr, label=f"MERIT-Net (AUC = {auc:.4f})")
plt.plot([0,1],[0,1], "--", color="grey")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curve"); plt.legend(loc="lower right")
plt.tight_layout(); plt.savefig("figures/fig6_roc_curve.png", dpi=300); plt.show()


## 6. Trust Score (Table 6 statistics, Figures 4 & 5)
Trust Score T = (0.4*A + 0.4*C + 0.2*V) * 100, with A = inter-model agreement,
C = ensemble confidence (max softmax), V = stability (low inter-model variance).
Component definitions follow the manuscript and `merit_net.py`.


In [ ]:
A = (((model_p >= 0.5).astype(int) == pred[:,None]).mean(1))   # agreement
C = np.maximum(ens_p, 1 - ens_p)                               # confidence
V = np.clip(1 - 2*model_p.std(1), 0, 1)                        # stability
T = (0.4*A + 0.4*C + 0.2*V) * 100
correct = (pred == y)

gap   = T[correct].mean() - T[~correct].mean()
auroc = roc_auc_score((~correct).astype(int), -T)             # error-detection AUROC
print(f"Trust Score  correct mean = {T[correct].mean():.2f} | incorrect mean = {T[~correct].mean():.2f}")
print(f"             gap = {gap:.2f} pp | error-detection AUROC = {auroc:.4f}")

# Figure 4 - distribution
plt.figure(figsize=(5,3.5))
plt.hist(T[correct],  bins=30, alpha=0.6, label="correct",   color="#2c7fb8")
plt.hist(T[~correct], bins=15, alpha=0.8, label="incorrect", color="#d95f0e")
plt.xlabel("Trust Score"); plt.ylabel("count"); plt.title("Trust Score distribution"); plt.legend()
plt.tight_layout(); plt.savefig("figures/fig4_trust_distribution.png", dpi=300); plt.show()

# Figure 5 - boxplot
plt.figure(figsize=(4,4))
plt.boxplot([T[correct], T[~correct]], labels=["correct","incorrect"])
plt.ylabel("Trust Score"); plt.title("Trust Score by prediction outcome")
plt.tight_layout(); plt.savefig("figures/fig5_trust_boxplot.png", dpi=300); plt.show()


## 7. Figure 3 — SHAP feature importance


In [ ]:
import shap
rf = RandomForestClassifier(n_estimators=NTREES, max_depth=20, random_state=SEED, class_weight={0:1.0,1:5.0}).fit(X, y)
expl = shap.TreeExplainer(rf)
sv = expl.shap_values(X)
if isinstance(sv, list):       sv1 = sv[1]           # old shap API: list per class
elif np.ndim(sv) == 3:         sv1 = sv[:, :, 1]     # new shap API: (n, features, classes)
else:                          sv1 = sv             # SHAP for the high-risk class
plt.figure()
shap.summary_plot(sv1, X, feature_names=FEATURES, show=False)
plt.tight_layout(); plt.savefig("figures/fig3_shap_summary.png", dpi=300, bbox_inches="tight"); plt.show()


## 8. Table 3 — Baselines
Run `baseline_configs.py` for the leakage-free re-implementation of the comparison methods:
```bash
python baseline_configs.py
```


---
Generated figures are written to `figures/`. All randomness is controlled by `random_state = 42`.
Confusion matrix and headline metrics reproduce the paper exactly at NTREES = 500.
